# Media Framing Normal API Run

This notebook prepares and runs the final thesis media-framing classification with the normal Responses API instead of the Batch API.

It keeps the same thesis logic as the batch notebook:
- one request per merged context window
- multiple media hits inside one window stay in the same row via `hit_text`
- `row_id` is preserved end-to-end
- Tagesschau is included, but direct Tagesschau self-references are removed
- the final result rows follow the same legacy schema as Katinka's earlier GPT run

The run cell is resumable and writes progress to CSV after every rate-limited chunk.


In [1]:
from __future__ import annotations

import json
import random
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from pandas.errors import EmptyDataError

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd() / '03_NER_Framing',
    Path.cwd() / '2a_NER',
    Path.cwd(),
    Path.cwd().parent / '03_NER_Framing',
    Path.cwd().parent / '2a_NER',
    Path('/Users/gretawette/Documents/Thesis/03_NER_Framing'),
    Path('/Users/MattisHaumann/Dev/Thesis/2a_NER'),
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / 'media_framing_batch_utils.py').exists()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate 2a_NER/media_framing_batch_utils.py from the current working directory.')

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from media_framing_batch_utils import (
    LEGACY_RESULT_COLUMNS,
    build_batch_requests_df,
    build_legacy_result_row,
    compile_master_pattern,
    estimate_runtime_from_rate_limits,
    estimate_tokens_heuristic,
    extract_media_contexts,
    read_env_value,
    split_requests_by_rate_budget,
    validate_batch_requests,
    write_manifest_csv,
)

DATA_PATH = PROJECT_ROOT / '01_EDAperOutlet' / 'outputs' / 'df_combined.csv'
PROMPT_PATH = NOTEBOOK_DIR / 'framing_codebook_prompt.txt'
OUTPUT_ROOT = NOTEBOOK_DIR / 'outputs' / 'batch_media_framing'
OUTPUT_DIR = OUTPUT_ROOT / 'final_thesis'
SYNC_MANIFEST_PATH = OUTPUT_DIR / 'media_framing_thesis_sync_manifest.csv'
SYNC_RESULTS_PATH = OUTPUT_DIR / 'media_framing_thesis_sync_results.csv'
SYNC_ERRORS_PATH = OUTPUT_DIR / 'media_framing_thesis_sync_errors.csv'
SYNC_RUNTIME_ESTIMATE_PATH = OUTPUT_DIR / 'media_framing_thesis_sync_runtime_estimate.csv'
BATCH_COMPARISON_MANIFEST_PATH = OUTPUT_DIR / 'media_framing_thesis_manifest.csv'

MODEL_NAME = 'gpt-5-mini'
WINDOW = 1
MAX_ALLOWED_ESTIMATED_HOURS = 10

STANDARD_INPUT_PRICE_PER_MILLION = 0.25
STANDARD_OUTPUT_PRICE_PER_MILLION = 2.00
OUTPUT_TOKEN_SCENARIOS = [40, 80]

# As of 2026-03-25, the official gpt-5-mini Tier-1 docs list 500 RPM and 500,000 TPM.
# This notebook intentionally stays below that. If your dashboard shows different limits, adjust these values first.
OPENAI_DOC_RPM_LIMIT = 500
OPENAI_DOC_TPM_LIMIT = 500_000
TARGET_REQUESTS_PER_MINUTE = 200
TARGET_INPUT_TOKENS_PER_MINUTE = 250_000
RUNTIME_OVERHEAD_FACTOR = 1.75
MAX_WORKERS = 16
REQUEST_TIMEOUT_SECONDS = 180
MAX_RETRIES = 6
BACKOFF_BASE_SECONDS = 2.0
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

FRAME_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'category': {
            'type': 'string',
            'enum': [
                'POSITIONS-/PARTEILICHKEITS-BIAS',
                'VERZERRUNG/MANIPULATION',
                'DISINFORMATION/FALSCHDARSTELLUNG',
                'VERSAGEN/INKOMPETENZ',
                'NEUTRAL',
                'IRRELEVANT',
            ],
        },
        'evidence': {'type': 'string'},
    },
    'required': ['category', 'evidence'],
}

ANALYSIS_INSTRUCTIONS = (
    'Return valid JSON that matches the schema exactly. '
    'Do not add any keys beyond category and evidence.'
)

for required_path in [DATA_PATH, PROMPT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required file not found: {required_path}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data path: {DATA_PATH}')
print(f'Prompt path: {PROMPT_PATH}')
print(f'Sync manifest path: {SYNC_MANIFEST_PATH}')
print(f'Sync results path: {SYNC_RESULTS_PATH}')


Data path: /Users/MattisHaumann/Dev/Thesis/2a_NER/df_combined.csv
Prompt path: /Users/MattisHaumann/Dev/Thesis/2a_NER/framing_codebook_prompt.txt
Sync manifest path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_sync_manifest.csv
Sync results path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_sync_results.csv


## 1. Load the Combined Data and the Codebook Prompt


In [2]:
df = pd.read_csv(DATA_PATH)

if 'row_id' not in df.columns:
    df = df.reset_index().rename(columns={'index': 'row_id'})

required_columns = {'row_id', 'source', 'Title', 'Text'}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f'df_combined.csv is missing required columns: {missing_columns}')

search_columns = [column for column in ['row_id', 'source', 'Date', 'Title', 'Text'] if column in df.columns]
search_df = df[search_columns].copy()
search_df['source'] = search_df['source'].fillna('').astype(str)

CODEBOOK_PROMPT = PROMPT_PATH.read_text(encoding='utf-8').strip()
if not CODEBOOK_PROMPT:
    raise ValueError(f'Prompt file is empty: {PROMPT_PATH}')

print(f'Articles loaded: {len(search_df):,}')
print(f"Unique sources: {search_df['source'].nunique():,}")
display(search_df.head(3))


Articles loaded: 20,440
Unique sources: 7


,row_id,source,Date,Title,Text
0,1,Antispiegel,2025-08-01,Bereitet der Westen die Entmachtung oder sogar...,Der Streit um das Nationale Anti-Korruptionsbü...
1,2,Antispiegel,2025-08-01,EU-Kommission hat Textnachrichten zum Kauf der...,Die New York Times versucht seit langem vor Ge...
2,3,Antispiegel,2025-08-01,Wahlkommission rechtfertigt Einmischung der EU...,Ende September stehen in Moldawien Parlamentsw...


## 2. Extract Context Windows and Run Thesis-Facing Checks

This applies the same mainstream-media filter and Tagesschau self-filter as the batch notebook.


In [3]:
MASTER_PATTERN = compile_master_pattern()

extraction = extract_media_contexts(search_df, pattern=MASTER_PATTERN, window=WINDOW)
media_context_df = extraction['media_context_df'].copy()
media_article_df = extraction['media_article_df'].copy()
kept_hits_df = extraction['kept_hits_df'].copy()
excluded_hits_df = extraction['excluded_hits_df'].copy()

required_context_columns = [
    'row_id',
    'source',
    'Title',
    'hit_text',
    'context_idx',
    'count_hits',
    'count_unique_entities',
    'context_window',
]
missing_context_columns = [column for column in required_context_columns if column not in media_context_df.columns]
if missing_context_columns:
    raise ValueError(f'media_context_df is missing required columns: {missing_context_columns}')
if media_context_df.empty:
    raise ValueError('media_context_df is empty after extraction.')
if media_context_df['row_id'].isna().any():
    raise AssertionError('row_id is missing in media_context_df.')
if media_context_df.duplicated(['row_id', 'context_idx']).any():
    raise AssertionError('row_id/context_idx pairs must be unique.')
if media_context_df['hit_text'].fillna('').eq('').any():
    raise AssertionError('Empty hit_text found in media_context_df.')

print(f"Candidate articles: {media_article_df['row_id'].nunique():,}")
print(f'Context rows ready: {len(media_context_df):,}')
print(f'Excluded self-reference hits: {len(excluded_hits_df):,}')
display(media_context_df.head(5))


Candidate articles: 7,204
Context rows ready: 11,975
Excluded self-reference hits: 7,213


,row_id,source,Title,Text,context_idx,context_window,hit_text,raw_hit_text,count_hits,count_unique_entities,Date
0,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Der Streit um das Nationale Anti-Korruptionsbü...,1,"Er hat es geschafft, seine Leute überall zu pl...",Politico,Politico,1,1,2025-08-01
1,4,Antispiegel,Fordert Russland wirklich die Vernichtung alle...,Ich bin auf die Geschichte zuerst bei RT-DE ge...,1,Viele haben Reitschuster aus der Corona-Zeit j...,Bild-Zeitung,Bild-Zeitung,1,1,2025-08-02
2,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,"Wer auch nicht-westliche Medien liest, der ist...",1,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,Der Spiegel,1,1,2025-08-02
3,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,"Wer auch nicht-westliche Medien liest, der ist...",2,Das NABU ist das vielleicht wichtigste Instrum...,Spiegel,Spiegel,1,1,2025-08-02
4,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,"Wer auch nicht-westliche Medien liest, der ist...",3,"Ich denke, die Antwort liegt auf der Hand. Mär...",Spiegel,Spiegel | der Spiegel,4,1,2025-08-02


## 3. Build the Normal-API Requests and Verify Legacy Result Shape

This keeps the exact same request body and final result schema as the batch workflow.


In [4]:
batch_requests_df = build_batch_requests_df(
    media_context_df,
    prompt_template=CODEBOOK_PROMPT,
    model_name=MODEL_NAME,
    analysis_instructions=ANALYSIS_INSTRUCTIONS,
    frame_schema=FRAME_SCHEMA,
)

validation_errors = validate_batch_requests(batch_requests_df)
if validation_errors:
    raise ValueError('Batch validation failed:\n' + '\n'.join(validation_errors[:20]))

manifest_df = batch_requests_df[
    [
        'custom_id',
        'hit_id',
        'row_id',
        'source',
        'Title',
        'hit_text',
        'context_idx',
        'count_hits',
        'count_unique_entities',
        'context_window',
    ]
].copy()

if manifest_df['row_id'].isna().any():
    raise AssertionError('row_id is missing in the manifest.')
if manifest_df['hit_id'].duplicated().any():
    raise AssertionError('hit_id values are not unique.')
if manifest_df['custom_id'].duplicated().any():
    raise AssertionError('custom_id values are not unique.')

smoke_result_row = build_legacy_result_row(
    manifest_df.iloc[0].to_dict(),
    {
        'id': 'resp_smoke_test',
        'model': MODEL_NAME,
        'output_text': json.dumps({'category': 'NEUTRAL', 'evidence': ''}),
    },
    default_model_name=MODEL_NAME,
)
if list(smoke_result_row.keys()) != LEGACY_RESULT_COLUMNS:
    raise AssertionError(list(smoke_result_row.keys()))

if BATCH_COMPARISON_MANIFEST_PATH.exists():
    existing_batch_manifest_df = pd.read_csv(BATCH_COMPARISON_MANIFEST_PATH)
    compare_columns = manifest_df.columns.tolist()
    left = manifest_df[compare_columns].sort_values('custom_id').reset_index(drop=True)
    right = existing_batch_manifest_df[compare_columns].sort_values('custom_id').reset_index(drop=True)
    pd.testing.assert_frame_equal(left, right, check_dtype=False)
    print(f'Compared successfully against existing batch manifest: {BATCH_COMPARISON_MANIFEST_PATH}')
else:
    print('No existing batch manifest found for comparison. Continuing with local validation only.')

print(f'Requests ready: {len(batch_requests_df):,}')
display(manifest_df.head(5))
display(pd.DataFrame([smoke_result_row]))


Compared successfully against existing batch manifest: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_manifest.csv
Requests ready: 11,975


,custom_id,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window
0,media-frame-d844fe7dbcb4b28a,d844fe7dbcb4b28a,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Politico,1,1,1,"Er hat es geschafft, seine Leute überall zu pl..."
1,media-frame-f785774b2205e418,f785774b2205e418,4,Antispiegel,Fordert Russland wirklich die Vernichtung alle...,Bild-Zeitung,1,1,1,Viele haben Reitschuster aus der Corona-Zeit j...
2,media-frame-8beba1b1263837c4,8beba1b1263837c4,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,1,1,1,Der Spiegel macht mal wieder Berichterstattung...
3,media-frame-db28e215f2d3989d,db28e215f2d3989d,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,2,1,1,Das NABU ist das vielleicht wichtigste Instrum...
4,media-frame-387a8dd7ac2cedde,387a8dd7ac2cedde,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,3,4,1,"Ich denke, die Antwort liegt auf der Hand. Mär..."


,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window,model,response_id,category,evidence,raw_response_json
0,d844fe7dbcb4b28a,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Politico,1,1,1,"Er hat es geschafft, seine Leute überall zu pl...",gpt-5-mini,resp_smoke_test,NEUTRAL,,"{""id"": ""resp_smoke_test"", ""model"": ""gpt-5-mini..."


## 4. Estimate Runtime and Standard API Cost

This notebook only proceeds if the conservative runtime estimate stays below 10 hours.


In [5]:
write_manifest_csv(batch_requests_df, SYNC_MANIFEST_PATH)

batch_requests_df = batch_requests_df.copy()
batch_requests_df['estimated_input_tokens'] = batch_requests_df['body'].map(
    lambda body: estimate_tokens_heuristic(json.dumps(body, ensure_ascii=False))
)
estimated_input_tokens_total = int(batch_requests_df['estimated_input_tokens'].sum())

runtime_estimate = estimate_runtime_from_rate_limits(
    n_requests=len(batch_requests_df),
    estimated_input_tokens=estimated_input_tokens_total,
    requests_per_minute_limit=TARGET_REQUESTS_PER_MINUTE,
    input_tokens_per_minute_limit=TARGET_INPUT_TOKENS_PER_MINUTE,
    overhead_factor=RUNTIME_OVERHEAD_FACTOR,
)
estimated_hours_with_overhead = runtime_estimate.estimated_minutes_with_overhead / 60

runtime_rows = [
    {
        'scope': 'full',
        'requests': len(batch_requests_df),
        'estimated_input_tokens': estimated_input_tokens_total,
        'doc_rpm_limit': OPENAI_DOC_RPM_LIMIT,
        'doc_tpm_limit': OPENAI_DOC_TPM_LIMIT,
        'target_requests_per_minute': TARGET_REQUESTS_PER_MINUTE,
        'target_input_tokens_per_minute': TARGET_INPUT_TOKENS_PER_MINUTE,
        'request_bound_minutes': round(runtime_estimate.request_bound_minutes, 2),
        'token_bound_minutes': round(runtime_estimate.token_bound_minutes, 2),
        'lower_bound_minutes': round(runtime_estimate.lower_bound_minutes, 2),
        'estimated_minutes_with_overhead': round(runtime_estimate.estimated_minutes_with_overhead, 2),
        'estimated_hours_with_overhead': round(estimated_hours_with_overhead, 2),
    }
]
runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(SYNC_RUNTIME_ESTIMATE_PATH, index=False, encoding='utf-8')

cost_rows = []
for output_tokens_per_request in OUTPUT_TOKEN_SCENARIOS:
    estimated_output_tokens = len(batch_requests_df) * output_tokens_per_request
    estimated_total_cost_usd = (
        estimated_input_tokens_total / 1_000_000 * STANDARD_INPUT_PRICE_PER_MILLION
        + estimated_output_tokens / 1_000_000 * STANDARD_OUTPUT_PRICE_PER_MILLION
    )
    cost_rows.append(
        {
            'assumed_output_tokens_per_request': output_tokens_per_request,
            'requests': len(batch_requests_df),
            'estimated_input_tokens': estimated_input_tokens_total,
            'estimated_output_tokens': estimated_output_tokens,
            'estimated_total_cost_usd': round(estimated_total_cost_usd, 4),
        }
    )
cost_estimate_df = pd.DataFrame(cost_rows)

print(f'Sync manifest written to: {SYNC_MANIFEST_PATH}')
print(f'Runtime estimate written to: {SYNC_RUNTIME_ESTIMATE_PATH}')
print(f'Conservative runtime estimate: {estimated_hours_with_overhead:.2f} hours')
print('This estimate is based on current local request volume and conservative targets below the published Tier-1 limits.')
display(runtime_df)
display(cost_estimate_df)

if estimated_hours_with_overhead > MAX_ALLOWED_ESTIMATED_HOURS:
    raise RuntimeError(
        f'Estimated runtime is {estimated_hours_with_overhead:.2f} hours, above the allowed limit of {MAX_ALLOWED_ESTIMATED_HOURS} hours.'
    )


Sync manifest written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_sync_manifest.csv
Runtime estimate written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_sync_runtime_estimate.csv
Conservative runtime estimate: 2.36 hours
This estimate is based on current local request volume and conservative targets below the published Tier-1 limits.


,scope,requests,estimated_input_tokens,doc_rpm_limit,doc_tpm_limit,target_requests_per_minute,target_input_tokens_per_minute,request_bound_minutes,token_bound_minutes,lower_bound_minutes,estimated_minutes_with_overhead,estimated_hours_with_overhead
0,full,11975,20266739,500,500000,200,250000,59.88,81.07,81.07,141.87,2.36


,assumed_output_tokens_per_request,requests,estimated_input_tokens,estimated_output_tokens,estimated_total_cost_usd
0,40,11975,20266739,479000,6.0247
1,80,11975,20266739,958000,6.9827


## 5. Optional: Run the Normal API with Resume Support

This cell resumes from the existing sync result file. Successful rows are never re-run. Failed rows can be retried.


In [ ]:
RUN_NORMAL_API = False
RETRY_PREVIOUS_ERROR_ROWS = True

def read_csv_or_empty(path: Path, *, columns=None) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        try:
            return pd.read_csv(path)
        except EmptyDataError:
            pass
    return pd.DataFrame(columns=columns) if columns else pd.DataFrame()

def build_error_row(row: dict, *, error_type: str, error_payload: str, status_code=None, raw_response_json: str = '') -> dict:
    return {
        'hit_id': row['hit_id'],
        'row_id': row['row_id'],
        'source': row['source'],
        'Title': row['Title'],
        'hit_text': row['hit_text'],
        'context_idx': row.get('context_idx', ''),
        'count_hits': row.get('count_hits', ''),
        'count_unique_entities': row.get('count_unique_entities', ''),
        'context_window': row['context_window'],
        'status_code': status_code,
        'error_type': error_type,
        'error_payload': error_payload,
        'raw_response_json': raw_response_json,
    }

def parse_retry_after_seconds(response, attempt: int) -> float:
    if response is not None:
        retry_after = response.headers.get('retry-after')
        if retry_after:
            try:
                return max(float(retry_after), 0.0)
            except ValueError:
                pass
    return BACKOFF_BASE_SECONDS * (2 ** attempt) + random.uniform(0.0, 0.5)

_thread_local = threading.local()

def get_session(api_key: str) -> requests.Session:
    session = getattr(_thread_local, 'session', None)
    if session is None:
        session = requests.Session()
        session.headers.update(
            {
                'Authorization': f'Bearer {api_key}',
                'Content-Type': 'application/json',
            }
        )
        _thread_local.session = session
    return session

def submit_request(row: dict, api_key: str):
    session = get_session(api_key)
    last_response_json = ''
    for attempt in range(MAX_RETRIES):
        try:
            response = session.post(
                'https://api.openai.com/v1/responses',
                json=row['body'],
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code in RETRYABLE_STATUS_CODES:
                try:
                    error_body = response.json()
                except ValueError:
                    error_body = {'message': response.text[:2000]}
                if attempt < MAX_RETRIES - 1:
                    time.sleep(parse_retry_after_seconds(response, attempt))
                    continue
                return False, None, build_error_row(
                    row,
                    error_type='http_error',
                    status_code=response.status_code,
                    error_payload=json.dumps(error_body, ensure_ascii=False),
                )
            response.raise_for_status()
            response_json = response.json()
            last_response_json = json.dumps(response_json, ensure_ascii=False)
            result_row = build_legacy_result_row(
                row,
                response_json,
                default_model_name=MODEL_NAME,
            )
            return True, result_row, None
        except requests.HTTPError as exc:
            response = exc.response
            status_code = response.status_code if response is not None else None
            if status_code in RETRYABLE_STATUS_CODES and attempt < MAX_RETRIES - 1:
                time.sleep(parse_retry_after_seconds(response, attempt))
                continue
            try:
                error_body = response.json() if response is not None else {'message': str(exc)}
            except ValueError:
                error_body = {'message': response.text[:2000] if response is not None else str(exc)}
            return False, None, build_error_row(
                row,
                error_type='http_error',
                status_code=status_code,
                error_payload=json.dumps(error_body, ensure_ascii=False),
            )
        except requests.RequestException as exc:
            if attempt < MAX_RETRIES - 1:
                time.sleep(parse_retry_after_seconds(None, attempt))
                continue
            return False, None, build_error_row(
                row,
                error_type='request_error',
                error_payload=str(exc),
            )
        except (json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
            return False, None, build_error_row(
                row,
                error_type='parse_error',
                error_payload=str(exc),
                raw_response_json=last_response_json,
            )

def save_progress(results_by_hit_id, errors_by_hit_id):
    results_df = pd.DataFrame(results_by_hit_id.values(), columns=LEGACY_RESULT_COLUMNS)
    if not results_df.empty:
        results_df = results_df.sort_values(
            ['row_id', 'context_idx', 'source', 'hit_text'],
            ascending=[True, True, True, True],
        ).drop_duplicates(subset=['hit_id'], keep='last').reset_index(drop=True)
    results_df.to_csv(SYNC_RESULTS_PATH, index=False, encoding='utf-8')

    errors_df = pd.DataFrame(errors_by_hit_id.values())
    if not errors_df.empty:
        sort_columns = [column for column in ['row_id', 'context_idx', 'source', 'hit_text'] if column in errors_df.columns]
        if sort_columns:
            errors_df = errors_df.sort_values(sort_columns).reset_index(drop=True)
    errors_df.to_csv(SYNC_ERRORS_PATH, index=False, encoding='utf-8')
    return results_df, errors_df

existing_results_df = read_csv_or_empty(SYNC_RESULTS_PATH, columns=LEGACY_RESULT_COLUMNS)
existing_errors_df = read_csv_or_empty(SYNC_ERRORS_PATH)

results_by_hit_id = {str(row['hit_id']): row for row in existing_results_df.to_dict('records')}
errors_by_hit_id = {str(row['hit_id']): row for row in existing_errors_df.to_dict('records')} if not existing_errors_df.empty and 'hit_id' in existing_errors_df.columns else {}

processed_hit_ids = set(results_by_hit_id)
if not RETRY_PREVIOUS_ERROR_ROWS:
    processed_hit_ids.update(errors_by_hit_id)

run_requests_df = batch_requests_df[~batch_requests_df['hit_id'].astype(str).isin(processed_hit_ids)].copy()
remaining_runtime_estimate = estimate_runtime_from_rate_limits(
    n_requests=len(run_requests_df),
    estimated_input_tokens=int(run_requests_df['estimated_input_tokens'].sum()) if not run_requests_df.empty else 0,
    requests_per_minute_limit=TARGET_REQUESTS_PER_MINUTE,
    input_tokens_per_minute_limit=TARGET_INPUT_TOKENS_PER_MINUTE,
    overhead_factor=RUNTIME_OVERHEAD_FACTOR,
)

print(f'Existing success rows: {len(existing_results_df):,}')
print(f'Existing error rows: {len(existing_errors_df):,}')
print(f'Rows remaining for this run: {len(run_requests_df):,}')
print(f'Estimated remaining runtime: {remaining_runtime_estimate.estimated_minutes_with_overhead / 60:.2f} hours')

if RUN_NORMAL_API and not run_requests_df.empty:
    api_key, api_key_source = read_env_value('OPENAI_API_KEY', project_root=PROJECT_ROOT)
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY not found. Add it to .env or export it in your shell.')
    if remaining_runtime_estimate.estimated_minutes_with_overhead / 60 > MAX_ALLOWED_ESTIMATED_HOURS:
        raise RuntimeError('Remaining runtime estimate exceeds the allowed 10-hour ceiling.')

    request_chunks = split_requests_by_rate_budget(
        run_requests_df,
        max_requests_per_window=TARGET_REQUESTS_PER_MINUTE,
        max_estimated_input_tokens_per_window=TARGET_INPUT_TOKENS_PER_MINUTE,
    )

    print(f'API key source: {api_key_source}')
    print(f'Chunks to process: {len(request_chunks):,}')
    run_started_at = time.monotonic()

    for chunk_idx, chunk_df in enumerate(request_chunks, start=1):
        chunk_started_at = time.monotonic()
        chunk_records = chunk_df.to_dict('records')
        chunk_input_tokens = int(chunk_df['estimated_input_tokens'].sum())

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(submit_request, row, api_key) for row in chunk_records]
            for future in as_completed(futures):
                ok, result_row, error_row = future.result()
                if ok and result_row is not None:
                    hit_id = str(result_row['hit_id'])
                    results_by_hit_id[hit_id] = result_row
                    errors_by_hit_id.pop(hit_id, None)
                elif error_row is not None:
                    errors_by_hit_id[str(error_row['hit_id'])] = error_row

        saved_results_df, saved_errors_df = save_progress(results_by_hit_id, errors_by_hit_id)
        chunk_elapsed_seconds = time.monotonic() - chunk_started_at
        total_elapsed_hours = (time.monotonic() - run_started_at) / 3600
        remaining_chunks = len(request_chunks) - chunk_idx
        sleep_seconds = max(0.0, 60.0 - chunk_elapsed_seconds)

        print(
            f'Chunk {chunk_idx}/{len(request_chunks)} done: '
            f'{len(chunk_records):,} requests, '
            f'{chunk_input_tokens:,} estimated input tokens, '
            f'{len(saved_results_df):,} success rows saved, '
            f'{len(saved_errors_df):,} error rows saved, '
            f'elapsed {total_elapsed_hours:.2f}h'
        )
        if remaining_chunks > 0 and sleep_seconds > 0:
            print(f'Sleeping {sleep_seconds:.1f}s to stay below the configured minute budget.')
            time.sleep(sleep_seconds)

    final_results_df, final_errors_df = save_progress(results_by_hit_id, errors_by_hit_id)
    print(f'Run complete. Final success rows: {len(final_results_df):,}')
    print(f'Final error rows: {len(final_errors_df):,}')
else:
    print('Normal API run disabled. Set RUN_NORMAL_API = True only when you intentionally want to execute the requests.')


Existing success rows: 0
Existing error rows: 0
Rows remaining for this run: 11,975
Estimated remaining runtime: 2.36 hours
API key source: environment variable
Chunks to process: 82
Chunk 1/82 done: 140 requests, 248,090 estimated input tokens, 140 success rows saved, 0 error rows saved, elapsed 0.02h
Chunk 2/82 done: 142 requests, 249,115 estimated input tokens, 282 success rows saved, 0 error rows saved, elapsed 0.05h
Chunk 3/82 done: 139 requests, 248,723 estimated input tokens, 421 success rows saved, 0 error rows saved, elapsed 0.07h
Chunk 4/82 done: 142 requests, 248,919 estimated input tokens, 563 success rows saved, 0 error rows saved, elapsed 0.09h
Chunk 5/82 done: 147 requests, 248,602 estimated input tokens, 710 success rows saved, 0 error rows saved, elapsed 0.11h
Chunk 6/82 done: 148 requests, 249,683 estimated input tokens, 858 success rows saved, 0 error rows saved, elapsed 0.13h
Chunk 7/82 done: 147 requests, 248,409 estimated input tokens, 1,005 success rows saved, 0 

## 6. Reload the Current Sync Output for Analysis

Run this after a finished or partial sync pass. The results CSV is already in the same legacy row shape as the batch parser output.


In [9]:
sync_results_df = pd.read_csv(SYNC_RESULTS_PATH) if SYNC_RESULTS_PATH.exists() and SYNC_RESULTS_PATH.stat().st_size > 0 else pd.DataFrame(columns=LEGACY_RESULT_COLUMNS)
sync_errors_df = pd.read_csv(SYNC_ERRORS_PATH) if SYNC_ERRORS_PATH.exists() and SYNC_ERRORS_PATH.stat().st_size > 0 else pd.DataFrame()

if not sync_results_df.empty:
    sync_results_df = sync_results_df[LEGACY_RESULT_COLUMNS].sort_values(
        ['row_id', 'context_idx', 'source', 'hit_text'],
        ascending=[True, True, True, True],
    ).drop_duplicates(subset=['hit_id'], keep='last').reset_index(drop=True)
    if sync_results_df['hit_id'].duplicated().any():
        raise AssertionError('Duplicate hit_id values found in sync results.')
    sync_results_df.to_csv(SYNC_RESULTS_PATH, index=False, encoding='utf-8')

summary_df = pd.DataFrame(
    [
        {
            'manifest_rows': len(manifest_df),
            'result_rows': len(sync_results_df),
            'error_rows': len(sync_errors_df),
            'remaining_rows_without_success': max(len(manifest_df) - len(sync_results_df), 0),
            'results_path': str(SYNC_RESULTS_PATH),
            'errors_path': str(SYNC_ERRORS_PATH),
        }
    ]
)

display(summary_df)
display(sync_results_df.head(5))
display(sync_errors_df.head(5))


,manifest_rows,result_rows,error_rows,remaining_rows_without_success,results_path,errors_path
0,11975,11973,2,2,/Users/MattisHaumann/Dev/Thesis/2a_NER/outputs...,/Users/MattisHaumann/Dev/Thesis/2a_NER/outputs...


,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window,model,response_id,category,evidence,raw_response_json
0,d844fe7dbcb4b28a,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Politico,1,1,1,"Er hat es geschafft, seine Leute überall zu pl...",gpt-5-mini,resp_0e3ba66f85f590500069c4635798988194bf13317...,NEUTRAL,NaN,"{""id"": ""resp_0e3ba66f85f590500069c463579898819..."
1,f785774b2205e418,4,Antispiegel,Fordert Russland wirklich die Vernichtung alle...,Bild-Zeitung,1,1,1,Viele haben Reitschuster aus der Corona-Zeit j...,gpt-5-mini,resp_05d159969059d7ad0069c4635784cc81938df21e9...,DISINFORMATION/FALSCHDARSTELLUNG,Propagandisten von Bild-Zeitung,"{""id"": ""resp_05d159969059d7ad0069c4635784cc819..."
2,8beba1b1263837c4,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,1,1,1,Der Spiegel macht mal wieder Berichterstattung...,gpt-5-mini,resp_084a85555116e7270069c4635799bc81949029652...,VERSAGEN/INKOMPETENZ,Berichterstattung für den Kindergarten,"{""id"": ""resp_084a85555116e7270069c4635799bc819..."
3,db28e215f2d3989d,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,2,1,1,Das NABU ist das vielleicht wichtigste Instrum...,gpt-5-mini,resp_067c239f074dd8230069c46357985c8193befbad9...,NEUTRAL,NaN,"{""id"": ""resp_067c239f074dd8230069c46357985c819..."
4,387a8dd7ac2cedde,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,3,4,1,"Ich denke, die Antwort liegt auf der Hand. Mär...",gpt-5-mini,resp_0f30c6857a242fb20069c463578cd0819496d5c8e...,POSITIONS-/PARTEILICHKEITS-BIAS,anti-russischen und pro-ukrainischen Propagand...,"{""id"": ""resp_0f30c6857a242fb20069c463578cd0819..."


,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window,status_code,error_type,error_payload,raw_response_json
0,97bf745e6b2d9b17,6902,RT_de,Werden Moskau und Washington Ukraine-Gespräche...,Politico,2,2,1,Ein namentlich nicht genannter US‑Beamter teil...,520,http_error,"{""message"": ""<!DOCTYPE html>\n<!--[if lt IE 7]...",NaN
1,1edca253aff4252e,6903,RT_de,Erste Reaktionen in Berlin auf US-Friedensplan...,RTL,1,1,1,Der Kanzleramtschef Thorsten Frei hat sich übe...,520,http_error,"{""message"": ""<!DOCTYPE html>\n<!--[if lt IE 7]...",NaN
